# Assignment 3: Public Tests

Contains one diagnostic test per question and carries no marks. Q1 checks the n-step A3C/actor-critic computation. Q2 checks the bounded continuous actor and one standard DDPG update.

In [ ]:
import json
from pathlib import Path
import torch
import numpy as np
import copy

SUBMISSION_NOTEBOOK = #TODO: Fill the path to your solution notebook.
ALLOWED_TAGS = {"provided", "graded_q1", "graded_q2"}


def load_submission(path=SUBMISSION_NOTEBOOK):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Place {SUBMISSION_NOTEBOOK} in the same directory as this test notebook."
        )
    notebook = json.loads(path.read_text(encoding="utf-8"))
    namespace = {}
    loaded = 0
    for cell in notebook["cells"]:
        tags = set(cell.get("metadata", {}).get("tags", []))
        if cell.get("cell_type") == "code" and tags.intersection(ALLOWED_TAGS):
            exec("".join(cell.get("source", [])), namespace)
            loaded += 1
    if loaded != 5:
        raise AssertionError("Required cell tags were changed or required cells were deleted.")
    return namespace


ns = load_submission()
globals().update(ns)

In [ ]:
# PUBLIC TEST Q1 — diagnostic only, 0 marks
set_seed = ns["set_seed"]
ActorCritic = ns["ActorCritic"]
compute_n_step_returns = ns["compute_n_step_returns"]
a3c_loss = ns["a3c_loss"]
a3c_worker_update = ns["a3c_worker_update"]

set_seed(11)
rewards = torch.tensor([1.0, 2.0])
dones = torch.tensor([0.0, 1.0])
returns = compute_n_step_returns(rewards, dones, torch.tensor(10.0), 0.9)
assert returns.dtype == torch.float32 and returns.shape == (2,)
assert torch.allclose(returns, torch.tensor([2.8, 2.0]), atol=1e-6)

global_model = ActorCritic(4, 2, hidden_dim=8)
local_model = ActorCritic(4, 2, hidden_dim=8)
optimizer = torch.optim.SGD(global_model.parameters(), lr=1e-3)
rollout = {
    "states": torch.tensor([[0.0, 1.0, 2.0, 3.0], [1.0, 0.0, 1.0, 2.0]]),
    "actions": torch.tensor([0, 1]),
    "rewards": rewards,
    "dones": dones,
    "bootstrap_value": torch.tensor(0.0),
}
metrics = a3c_worker_update(global_model, local_model, optimizer, rollout)
assert set(metrics) == {
    "total_loss",
    "policy_loss",
    "value_loss",
    "entropy",
    "grad_norm",
}
assert all(np.isfinite(v) and isinstance(v, float) for v in metrics.values())
print("PASS Q1 public test")

In [ ]:
# PUBLIC TEST Q2 — diagnostic only, 0 marks
Actor = ns["Actor"]
Critic = ns["Critic"]
soft_update = ns["soft_update"]
ddpg_update = ns["ddpg_update"]

set_seed(12)
low = np.array([-2.0, -1.0], dtype=np.float32)
high = np.array([2.0, 3.0], dtype=np.float32)
actor = Actor(3, low, high, hidden_dim=8)
critic = Critic(3, 2, hidden_dim=8)
target_actor = copy.deepcopy(actor)
target_critic = copy.deepcopy(critic)
states = torch.randn(4, 3)
actions = actor(states)
assert actions.shape == (4, 2)
assert torch.all(actions >= torch.tensor(low) - 1e-6)
assert torch.all(actions <= torch.tensor(high) + 1e-6)
assert critic(states, actions).shape == (4, 1)

batch = {
    "states": states,
    "actions": actions.detach(),
    "rewards": torch.tensor([[1.0], [0.0], [-1.0], [2.0]]),
    "next_states": torch.randn(4, 3),
    "dones": torch.tensor([[0.0], [1.0], [0.0], [1.0]]),
}
actor_optimizer = torch.optim.Adam(actor.parameters(), lr=1e-3)
critic_optimizer = torch.optim.Adam(critic.parameters(), lr=1e-3)
metrics = ddpg_update(
    actor,
    critic,
    target_actor,
    target_critic,
    actor_optimizer,
    critic_optimizer,
    batch,
    gamma=0.9,
    tau=0.05,
)
assert set(metrics) == {"critic_loss", "actor_loss", "mean_target_q"}
assert all(np.isfinite(v) and isinstance(v, float) for v in metrics.values())
print("PASS Q2 public test")